## 1. Import Libraries

In [7]:
import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

from scipy import stats
from scipy.stats.mstats import winsorize

import warnings
warnings.filterwarnings("ignore")

## 2. Load Dataset

In [8]:
print("Load Dataset")

df_original = pd.read_csv(r"C:\Users\tanaa\Downloads\patient_data.csv")

df_original.head()

Load Dataset


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,NaN,NaN,NaN,NaN,133.744915,NaN,NaN,1
1,2,71.0,Male,North,26.886502,250.394654,170.879882,124.536690,0
2,3,NaN,NaN,NaN,NaN,134.970152,NaN,NaN,0
3,4,34.0,Female,North,20.389173,76.556169,254.995057,72.938453,1
4,5,62.0,Male,East,29.348030,151.325621,174.161433,73.910108,0


## 3. Data Understanding

In [9]:
df = df_original.copy()

print("\n--- INFO ---")
print(df.info())

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Statistical Summary ---")
print(df.describe())


--- INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      500 non-null    int64  
 1   age             450 non-null    float64
 2   gender          450 non-null    object 
 3   region          450 non-null    object 
 4   bmi             450 non-null    float64
 5   blood_pressure  500 non-null    float64
 6   cholesterol     450 non-null    float64
 7   glucose         450 non-null    float64
 8   disease_risk    500 non-null    int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 35.3+ KB
None

--- Missing Values ---
patient_id         0
age               50
gender            50
region            50
bmi               50
blood_pressure     0
cholesterol       50
glucose           50
disease_risk       0
dtype: int64

--- Statistical Summary ---
       patient_id         age         bmi  blood_pressure  chole

## 4. Missing Values (%)

In [10]:
print("\n--- Missing Percentage ---")
print(df.isnull().mean() * 100)


--- Missing Percentage ---
patient_id         0.0
age               10.0
gender            10.0
region            10.0
bmi               10.0
blood_pressure     0.0
cholesterol       10.0
glucose           10.0
disease_risk       0.0
dtype: float64


# PART A: Handling Missing Values

### 4.1 Simple Imputer

In [11]:
df_simple = df.copy()

# Numerical columns
num_cols = ["bmi", "cholesterol", "glucose"]

df_simple[num_cols] = SimpleImputer(strategy="mean").fit_transform(df_simple[num_cols])

print("Numerical columns filled with MEAN")


# Categorical columns
df_simple["gender"] = SimpleImputer(strategy="most_frequent").fit_transform(df_simple[["gender"]]).ravel()
df_simple["region"] = SimpleImputer(strategy="most_frequent").fit_transform(df_simple[["region"]]).ravel()

print("Categorical columns filled with MODE")

print("\nMissing After Simple Imputer:")
print(df_simple.isnull().sum())

Numerical columns filled with MEAN
Categorical columns filled with MODE

Missing After Simple Imputer:
patient_id         0
age               50
gender             0
region             0
bmi                0
blood_pressure     0
cholesterol        0
glucose            0
disease_risk       0
dtype: int64


### 4.2 KNN Imputer

In [12]:
df_knn = df.copy()

num_cols = df_knn.select_dtypes(include=['float64', 'int64']).columns

knn = KNNImputer(n_neighbors=5)
df_knn[num_cols] = knn.fit_transform(df_knn[num_cols])

print("After KNN Imputation:\n", df_knn.isnull().sum())

After KNN Imputation:
 patient_id         0
age                0
gender            50
region            50
bmi                0
blood_pressure     0
cholesterol        0
glucose            0
disease_risk       0
dtype: int64


### 4.3 MICE (Best Method)

In [13]:
df_mice = df.copy()

num_cols = df_mice.select_dtypes(include=['float64', 'int64']).columns

mice = IterativeImputer()
df_mice[num_cols] = mice.fit_transform(df_mice[num_cols])

print("After MICE Imputation:\n", df_mice.isnull().sum())

After MICE Imputation:
 patient_id         0
age                0
gender            50
region            50
bmi                0
blood_pressure     0
cholesterol        0
glucose            0
disease_risk       0
dtype: int64


# PART B: Handling Outliers

## 5.1 Z-Score Method

In [14]:
print("\n--- Z-Score Method ---")

df_out = df_mice.copy()

z = stats.zscore(df_out[["cholesterol", "glucose"]])

df_z = df_out.copy()

df_z["cholesterol"] = df_z["cholesterol"].where(abs(z[:,0]) < 3, df_z["cholesterol"].median())
df_z["glucose"] = df_z["glucose"].where(abs(z[:,1]) < 3, df_z["glucose"].median())

df_z.head()


--- Z-Score Method ---


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1.0,50.198049,NaN,NaN,28.446418,133.744915,216.012006,110.514990,1.0
1,2.0,71.000000,Male,North,26.886502,250.394654,170.879882,124.536690,0.0
2,3.0,50.200043,NaN,NaN,28.446020,134.970152,215.970733,110.511749,0.0
3,4.0,34.000000,Female,North,20.389173,76.556169,254.995057,72.938453,1.0
4,5.0,62.000000,Male,East,29.348030,151.325621,174.161433,73.910108,0.0


## 5.2 IQR Method

In [15]:
print("\n--- IQR Method ---")

df_iqr = df_out.copy()

Q1 = df_iqr["bmi"].quantile(0.25)
Q3 = df_iqr["bmi"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df_iqr["bmi"] = df_iqr["bmi"].clip(lower, upper)

df_iqr.head()


--- IQR Method ---


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1.0,50.198049,NaN,NaN,28.446418,133.744915,216.012006,110.514990,1.0
1,2.0,71.000000,Male,North,26.886502,250.394654,170.879882,124.536690,0.0
2,3.0,50.200043,NaN,NaN,28.446020,134.970152,215.970733,110.511749,0.0
3,4.0,34.000000,Female,North,20.389173,76.556169,254.995057,72.938453,1.0
4,5.0,62.000000,Male,East,29.348030,151.325621,174.161433,73.910108,0.0


## 5.3 Percentile Method

In [16]:
print("\n--- Percentile Method ---")

df_pct = df_out.copy()

lower = df_pct["glucose"].quantile(0.01)
upper = df_pct["glucose"].quantile(0.99)

df_pct["glucose"] = df_pct["glucose"].clip(lower, upper)

df_pct.head()


--- Percentile Method ---


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1.0,50.198049,NaN,NaN,28.446418,133.744915,216.012006,110.514990,1.0
1,2.0,71.000000,Male,North,26.886502,250.394654,170.879882,124.536690,0.0
2,3.0,50.200043,NaN,NaN,28.446020,134.970152,215.970733,110.511749,0.0
3,4.0,34.000000,Female,North,20.389173,76.556169,254.995057,72.938453,1.0
4,5.0,62.000000,Male,East,29.348030,151.325621,174.161433,73.910108,0.0


## 5.4 Winsorization (BEST)

In [17]:
print("\n--- Winsorization Method ---")

df_win = df_out.copy()

df_win["cholesterol"] = winsorize(df_win["cholesterol"], limits=[0.01, 0.01])
df_win["glucose"] = winsorize(df_win["glucose"], limits=[0.01, 0.01])

df_win.head()


--- Winsorization Method ---


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1.0,50.198049,NaN,NaN,28.446418,133.744915,216.012006,110.514990,1.0
1,2.0,71.000000,Male,North,26.886502,250.394654,170.879882,124.536690,0.0
2,3.0,50.200043,NaN,NaN,28.446020,134.970152,215.970733,110.511749,0.0
3,4.0,34.000000,Female,North,20.389173,76.556169,254.995057,72.938453,1.0
4,5.0,62.000000,Male,East,29.348030,151.325621,174.161433,73.910108,0.0


## 6. Compare Results

In [18]:
print("\nOriginal:\n", df_out.describe())

print("\nZ-score:\n", df_z.describe())

print("\nIQR:\n", df_iqr.describe())

print("\nPercentile:\n", df_pct.describe())

print("\nWinsorization:\n", df_win.describe())


Original:
        patient_id         age         bmi  blood_pressure  cholesterol  \
count  500.000000  500.000000  500.000000      500.000000   500.000000   
mean   250.500000   50.646693   28.361046      126.982930   211.580230   
std    144.481833   16.562465   12.723213       31.293580    60.113371   
min      1.000000   20.000000   10.757287       76.179743    79.219514   
25%    125.750000   39.000000   22.681841      110.398487   180.687084   
50%    250.500000   50.919923   26.397475      121.943380   207.645950   
75%    375.250000   63.000000   28.720057      133.714056   226.827819   
max    500.000000   79.000000  111.829502      306.315690   509.892684   

          glucose  disease_risk  
count  500.000000    500.000000  
mean   110.116426      0.528000  
std     52.975370      0.499715  
min     20.927568      0.000000  
25%     82.052844      0.000000  
50%    107.871066      1.000000  
75%    120.527698      1.000000  
max    482.124560      1.000000  

Z-score:
     

# PART C: Final Clean Dataset

In [19]:
df_final = df_win

print("\n--- Final Dataset Check ---")
print("Missing Values:\n", df_final.isnull().sum())

print("\nFinal Summary:\n", df_final.describe())
df_final.to_csv


--- Final Dataset Check ---
Missing Values:
 patient_id         0
age                0
gender            50
region            50
bmi                0
blood_pressure     0
cholesterol        0
glucose            0
disease_risk       0
dtype: int64

Final Summary:
        patient_id         age         bmi  blood_pressure  cholesterol  \
count  500.000000  500.000000  500.000000      500.000000   500.000000   
mean   250.500000   50.646693   28.361046      126.982930   211.646655   
std    144.481833   16.562465   12.723213       31.293580    58.954286   
min      1.000000   20.000000   10.757287       76.179743   113.093218   
25%    125.750000   39.000000   22.681841      110.398487   180.687084   
50%    250.500000   50.919923   26.397475      121.943380   207.645950   
75%    375.250000   63.000000   28.720057      133.714056   226.827819   
max    500.000000   79.000000  111.829502      306.315690   486.403845   

          glucose  disease_risk  
count  500.000000    500.000000  


<bound method NDFrame.to_csv of      patient_id        age  gender region        bmi  blood_pressure  \
0           1.0  50.198049     NaN    NaN  28.446418      133.744915   
1           2.0  71.000000    Male  North  26.886502      250.394654   
2           3.0  50.200043     NaN    NaN  28.446020      134.970152   
3           4.0  34.000000  Female  North  20.389173       76.556169   
4           5.0  62.000000    Male   East  29.348030      151.325621   
..          ...        ...     ...    ...        ...             ...   
495       496.0  51.071867     NaN    NaN  28.280044      133.559023   
496       497.0  44.000000  Female   West  28.663200       84.556013   
497       498.0  51.110767     NaN    NaN  28.272868      104.854039   
498       499.0  32.000000    Male   West  25.393176      129.287314   
499       500.0  79.000000    Male   East  15.008997      150.862432   

     cholesterol     glucose  disease_risk  
0     216.012006  110.514990           1.0  
1     170.879

# 7. Save Final Dataset

In [29]:
import os

# create folder if not exists
os.makedirs(r"C:\Users\tanaa\Downloads\data", exist_ok=True)

# now save file
df_final.to_csv(r"C:\Users\tanaa\Downloads\data\cleaned_patient_data.csv", index=False)

print("Dataset Saved Successfully")

Dataset Saved Successfully
